# FARUMASI shift dispatcher — T4 sweep

Runtime: **GPU → T4**. This notebook trains DQN, REINFORCE, PPO, and A2C for 100,000 frames each, across 10 hyperparameter settings (40 runs). The environment is the same morning pharmacy shift in every run. A finished run is skipped if you start the cell again, so a disconnect keeps the experiments that already completed.

Opening this notebook from GitHub loads only the notebook. The next cell clones the repository when `main.py` is not already in the working directory. Use a **T4** runtime. Colab already has a CUDA build of PyTorch; the install cell leaves that wheel in place.

In [ ]:
!nvidia-smi
import torch
print("cuda", torch.cuda.is_available(), getattr(torch.version, "cuda", None), torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit("No GPU yet. Runtime → Change runtime type → T4, then Runtime → Restart session, and run this cell again.")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/Emmanuel-NS/Rl_Mission-based_Summative.git"
root = Path.cwd()
candidates = [
    root,
    root / "Rl_Mission-based_Summative",
    root / "emmanuel_nsabagasani_rl_summative",
    Path("/content/Rl_Mission-based_Summative"),
]
for folder in candidates:
    if (folder / "main.py").exists():
        os.chdir(folder)
        break
else:
    dest = Path("/content/Rl_Mission-based_Summative")
    if dest.exists():
        subprocess.check_call(["git", "-C", str(dest), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(dest)])
    os.chdir(dest)
print("working directory", Path.cwd())

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "CUDA torch is gone. Runtime → Restart session (this restores Colab's GPU build), "
        "confirm the runtime is T4, then run from the first cell. Do not pip install torch."
    )
print("keeping", torch.__version__, "cuda", torch.version.cuda)
# torchrl depends on torch. Installing it normally replaces Colab's CUDA wheel
# with the PyPI build, and torch.cuda.is_available() becomes False.
!pip install -q --upgrade-strategy only-if-needed "gymnasium>=1.0" pygame matplotlib "numpy>=1.26" cloudpickle packaging pyvers
!pip install -q --no-deps "torchrl==0.15.0" "tensordict==0.15.6" "hoptorch==0.1.4"
import torch
import gymnasium
import tensordict
import torchrl
assert torch.cuda.is_available(), "CUDA torch was replaced. Restart the session and run from the first cell."
print("ready", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
!python main.py --baselines
!python main.py --screenshot

In [ ]:
# 40 runs. On a T4 this is usually about one to two hours.
!python main.py --sweep all --device cuda --frames 100000 --eval-every 10000

In [ ]:
import shutil
from pathlib import Path
shutil.make_archive("/content/farumasi_rl_results", "zip", ".", ".")
print("Download farumasi_rl_results.zip and keep models/best_agent.pt, logs/, and assets/environment.png.")
try:
    from google.colab import files
    files.download("/content/farumasi_rl_results.zip")
except Exception as error:
    print(error)